# OP-08 · Calibration des hindcasts et comparaison brut ↔ calibré

**Notebook opérationnel** — à exécuter après OP-07 (skill brut), à chaque cycle.

| | |
|---|---|
| Étape du workflow | E5 (calibration, Draft §3.7) |
| Entrées | cumuls C3S par période, archives d'observation à 0,25° |
| Sorties | `OUTPUTS_OSF/calibrated/<cycle>/` (hindcasts calibrés), `OUTPUTS_OSF/skill/<cycle>/calibrated/` (scores, gains, cartes, diagrammes), `OUTPUTS_OSF/registry/calibration_methods.csv` |
| Durée indicative | ~5 h pour 8 modèles × mois et saisons × 10 méthodes (spectre de développement) |

Ce que fait cette étape, dans l'ordre :

1. **ramène le modèle sur la grille d'observation** (0,25°, CHIRPS pour la pluie) par
   interpolation bilinéaire — ce n'est pas une descente d'échelle, c'est la référence que le
   downscaling de E7 devra battre ;
2. **ajuste chaque méthode en LOYO** et prédit chaque année hors échantillon ;
3. **note chaque produit** avec la métrique qui lui sied, exactement comme en phase P2 — même
   fonction, même contexte d'observation, mêmes périodes, même masque ;
4. **compare au brut maille par maille** et en déduit la méthode recommandée par produit.

**Une méthode n'est essayée que sur les produits auxquels elle peut répondre.** La régression
logistique par catégorie ne rend que les trois catégories des terciles : elle ne peut pas
répondre à « plus de 300 mm » ni rendre un cumul. L'ELR décrit la distribution par ses seuils,
ce qui en fait un mauvais outil pour une valeur. Les corrections de biais, EQM, QDM et NGR/EMOS
servent tout. Le brut est toujours dans la liste : c'est la référence du gain.


## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
SYSTEMS      = ["c3s"]
VARIABLES    = ["precip"]          # la température viendra après la pluie
MODELS       = None                # None = tous les modèles
SCALES       = ["month", "season"] # les décades sont possibles, mais réservées à l'opérationnel
PERIODS      = None                # None = sous-ensemble de développement ; "all", "all-seasons"…
FAMILLES     = ["raw", "bias", "qmap", "logistic", "ngr"]
METRIQUES    = "deciding"          # métrique décisive + version débiaisée + discrimination
MIN_AMELIORE = 0.5                 # part du masque qu'une méthode doit améliorer


In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
OUT = cfg.output_root / "skill" / cfg.cycle_id / "calibrated"
REG = cfg.path_of("output_root") / "registry"
print("cycle", cfg.cycle_id, "| sorties", OUT)


## 1. Calibration, scores et comparaison

Une seule commande fait les quatre opérations. Elle écrit, pour chaque modèle, échelle,
produit et méthode : les scores (`netcdf/…/<produit>/<méthode>/<métrique>.nc`), la **carte de
gain** (`gain_<métrique>.nc`), la synthèse et le registre.

In [ ]:
from eccas_s2s.operations import calibrate_hindcast

ctx = calibrate_hindcast.run(CYCLE_CONFIG, systems=SYSTEMS, variables=VARIABLES,
                             models=MODELS, scales=SCALES, families=FAMILLES,
                             selection=PERIODS, metrics=METRIQUES,
                             min_improved=MIN_AMELIORE)
print(f"\nStatut : {ctx.status} — {len(ctx.outputs)} sortie(s)")


## 2. Le registre des méthodes recommandées

Une ligne par modèle × échelle × produit. Une méthode est **candidate** si elle améliore la
maille typique *et* au moins la moitié du masque ; la retenue est celle du plus fort gain
médian, départagée par la surface réparée. Si aucune ne qualifie, la recommandation est `raw`
et la colonne `reason` dit pourquoi.

In [ ]:
registre = pd.read_csv(REG / "calibration_methods.csv")
print(registre["method"].value_counts().to_string())
display(registre[registre.scale == "season"]
        .pivot_table(index="product", columns="model", values="method", aggfunc="first"))


## 3. Ce que la calibration apporte, produit par produit

`median_gain` est la médiane du gain **par maille**, pas la différence des médianes : une
méthode qui répare une région et en casse une autre ne peut pas s'y cacher. `fraction_repaired`
compte les mailles où le produit passe du côté utilisable, `fraction_broken` l'inverse.

In [ ]:
vue = registre.groupby("product").agg(
    methode=("method", lambda s: s.mode().iat[0]),
    gain_median=("median_gain", "median"),
    surface_amelioree=("fraction_improved", "median"),
    surface_reparee=("fraction_repaired", "median"),
    surface_cassee=("fraction_broken", "median"),
    laisse_brut=("method", lambda s: (s == "raw").sum()))
display(vue.round(3))


## 4. Cartes calibrées et cartes de gain

Par défaut, seules la **métrique décisive** et la **méthode recommandée** sont tracées : c'est
la carte qu'un prévisionniste lit. Les cartes de gain (`--kind gain`) montrent *où* la
calibration agit ; elles sont produites ici pour les **terciles**, produit phare de la prévision
saisonnière.

In [ ]:
from eccas_s2s.operations import plot_skill_raw

cal = plot_skill_raw.run(CYCLE_CONFIG, kind="calibrated", systems=SYSTEMS, variables=VARIABLES,
                         models=MODELS)
gain = plot_skill_raw.run(CYCLE_CONFIG, kind="gain", systems=SYSTEMS, variables=VARIABLES,
                          models=MODELS, products=["terciles"])
print(f"cartes calibrées : {len(cal.outputs)} | cartes de gain : {len(gain.outputs)}")


## 5. Diagrammes calibrés

Pour chaque famille (terciles, classes SPI, seuils de percentile, seuils de cumul), la méthode
recommandée par la majorité de ses produits est **réajustée** — une méthode au lieu de dix — et
ses probabilités alimentent les mêmes figures que le brut. Les deux jeux sont donc superposables.

In [ ]:
from eccas_s2s.operations import skill_diagrams

diag = skill_diagrams.run(CYCLE_CONFIG, kind="calibrated", systems=SYSTEMS,
                          variables=VARIABLES, models=MODELS, scales=SCALES,
                          selection=PERIODS)
print(f"Statut : {diag.status} — {len(diag.outputs)} sortie(s)")


## 6. Contrôle visuel

In [ ]:
from IPython.display import Image, display
from eccas_s2s.operations.skill_raw import score_paths

model, scale, produit = "dwd", "season", "terciles"
methode = registre[(registre.model == model) & (registre.scale == scale)
                   & (registre["product"] == produit)]["method"].iat[0]
periode = "season_m1"
for quoi in (f"{methode}/rpss", f"{methode}/gain_rpss"):
    f = score_paths(OUT / "figures", "c3s", model, "precip", scale, produit) / quoi / f"{periode}.png"
    if f.exists():
        display(Image(filename=str(f), width=520))


## 7. Ce qui est archivé

* `OUTPUTS_OSF/calibrated/<cycle>/…/<méthode>.nc` — la prévision calibrée elle-même (moyenne,
  probabilités des terciles sur les seuils observés, dispersion) ;
* `skill/<cycle>/calibrated/netcdf/…/<produit>/<méthode>/` — scores **et** cartes de gain ;
* `comparison_raw_calibrated.csv` — une ligne par produit, métrique, méthode et période, avec
  les quatre indicateurs spatiaux ;
* `registry/calibration_methods.csv` — la méthode retenue et pourquoi.

**Étape suivante :** la pondération multimodèle (P6) lira ce registre pour savoir, produit par
produit, quelle version de chaque modèle entre dans le mélange.
